# Classificação de Retinopatia Hipertensiva em Imagens de Fundo de Olho

Pipeline de aprendizado profundo para detecção automática de retinopatia hipertensiva em imagens de fundoscopia ocular, combinando os datasets BRSET e ODIR-5K.

Repositório: https://github.com/Bappoz/fundus-classification.git

| Atributo | Descrição |
|---|---|
| **Tarefa** | Classificação binária supervisionada: Normal vs. Retinopatia Hipertensiva |
| **Abordagem** | Transferência de aprendizado com backbone pré-treinado em ImageNet |
| **Métrica principal** | AUC-ROC (priorizada dado o desbalanceamento severo de classes) |
| **Plataforma** | Google Colab (GPU T4 / A100) |

---

**Sumário**
1. [Configuração do Ambiente](#1-configuracao)
2. [Verificação de Hardware](#2-hardware)
3. [Análise Exploratória dos Dados (EDA)](#3-eda)
   - 3.1 Inventário do Dataset
   - 3.2 Estrutura por Paciente e Data Leakage
   - 3.3 Distribuição de Classes e Fontes
   - 3.4 Amostras Visuais

---
## 1. Configuração do Ambiente

Detecta automaticamente o ambiente de execução (Colab vs. local), monta o Google Drive, clona ou atualiza o repositório e instala as dependências do projeto.

### Datasets Utilizados

Este projeto combina dois conjuntos públicos de fundoscopia ocular, selecionando apenas as classes de interesse:

| Dataset | Fonte | Papel no estudo |
|---|---|---|
| **BRSET** | Brazilian Multilabel Ophthalmological Dataset (USP) | Classe normal (8 457 imagens) + retinopatia hipertensiva (284 imagens) |
| **ODIR-5K** | Ocular Disease Intelligent Recognition Challenge | Retinopatia hipertensiva adicional (193 imagens) |

A combinação aumenta marginalmente a representação da classe positiva, mas o desbalanceamento permanece severo (~18:1). O impacto dessa adição será avaliado na etapa de treinamento.

#### Como obter o dataset

**Opção A — Colaborador do projeto**

Solicite o arquivo `dataset.zip` ao orientador e salve-o em `Meu Drive/`.

**Opção B — Download público**

1. Baixe o `.zip` (metadados + imagens BRSET e ODIR-5K): https://drive.google.com/file/d/1LL-8X1uCwgXRu0F_u2Q8mNDxEM2ylPcn/view?usp=sharing
2. Faça upload para `Meu Drive/` com o nome `dataset.zip`.

#### Estrutura interna esperada do `dataset.zip`

```
dataset.zip
└── retino/
    ├── meta/
    │   ├── labels_brset_filt.xlsx
    │   └── data_filt.xlsx
    └── images/
        ├── normal/
        ├── hr_brset/
        └── hr_odir5k/
```

Na primeira execução o zip é extraído para o Drive. Nas execuções seguintes a extração é ignorada automaticamente.

> **Nome do arquivo diferente?** Edite a variável `DRIVE_ZIP` no início da próxima célula.

In [ ]:
import os, sys

# ─── CONFIG ───────────────────────────────────────────────────────────────────
DRIVE_ZIP  = "/content/drive/MyDrive/dataset.zip"
DRIVE_ROOT = "/content/drive/MyDrive/retino"
REPO       = "https://github.com/Bappoz/fundus-classification.git"
# ──────────────────────────────────────────────────────────────────────────────

try:
    from google.colab import drive
    IN_COLAB = True
except ImportError:
    print("Ambiente local detectado — pulando mount do Drive.")
    IN_COLAB = False

if IN_COLAB:
    drive.mount("/content/drive")

    meta_ok   = os.path.isfile(f"{DRIVE_ROOT}/meta/labels_brset_filt.xlsx")
    images_ok = os.path.isdir(f"{DRIVE_ROOT}/images/normal")

    if meta_ok and images_ok:
        print(f"Dataset ja extraido em {DRIVE_ROOT} — pulando unzip.")
    elif os.path.isfile(DRIVE_ZIP):
        print(f"Extraindo {DRIVE_ZIP} para o Drive (apenas na primeira vez)...")
        !unzip -q -o "{DRIVE_ZIP}" -d "/content/drive/MyDrive/"
        print(f"Dataset extraido em {DRIVE_ROOT}.")
    else:
        print(f"Arquivo nao encontrado: {DRIVE_ZIP}")
        print("Verifique a variavel DRIVE_ZIP ou consulte a celula de documentacao acima.")

    # Repositório
    %cd /content
    if not os.path.isdir("/content/fundus-classification"):
        !git clone {REPO}
    else:
        print("Repositorio ja clonado — atualizando...")
        !cd /content/fundus-classification && git pull
    %cd /content/fundus-classification
    sys.path.insert(0, os.path.abspath("src"))
else:
    src_path = os.path.abspath(os.path.join(os.getcwd(), "..", "src"))
    if src_path not in sys.path:
        sys.path.insert(0, src_path)

if IN_COLAB:
    %pip install -q -e .
else:
    %pip install -q -e ..

%pip install -q timm albumentations
print("Ambiente configurado com sucesso.")

---
## 2. Verificação de Hardware

Confirma a disponibilidade de GPU e exibe a configuração do experimento. O treinamento de redes convolucionais profundas sobre imagens de alta resolução é computacionalmente inviável sem aceleração por GPU. Todos os experimentos deste trabalho foram executados em GPU NVIDIA T4 ou A100 via Google Colab.

> Sem GPU disponível: acesse *Runtime → Change runtime type → GPU (T4)* antes de prosseguir.

In [ ]:
import torch
from retino.config import cfg

gpu_ok      = torch.cuda.is_available()
device_name = torch.cuda.get_device_name(0) if gpu_ok else "N/A"

print(f"GPU disponivel : {'Sim' if gpu_ok else 'Nao'}")
print(f"  Dispositivo  : {device_name}")
print(f"  Device cfg   : {cfg.device}")
print(f"  Backbone     : {cfg.backbone}")

if not gpu_ok:
    raise RuntimeError("GPU nao disponivel — altere o runtime antes de prosseguir.")

---
## 3. Análise Exploratória dos Dados (EDA)

A análise exploratória tem três objetivos principais: (1) quantificar o desbalanceamento de classes para dimensionar as estratégias de mitigação; (2) identificar imagens de baixa qualidade que possam introduzir ruído no treinamento; e (3) inspecionar visualmente amostras de cada classe para verificar a diversidade e coerência clínica dos dados.

### 3.1 Inventário do Dataset

Carrega os metadados de ambas as fontes, verifica a integridade dos arquivos em disco e aplica o filtro de qualidade do BRSET.

In [ ]:
from retino.data.loader import build_labels, filter_quality, verify_files
from retino.config import cfg
from pathlib import Path

# Sobrescreve os paths para o ambiente Colab/Drive
cfg.meta_dir  = Path("/content/drive/MyDrive/retino/meta")
cfg.image_dir = Path("/content/drive/MyDrive/retino/images")

df = build_labels()
df = verify_files(df)
df = filter_quality(df)

n_total  = len(df)
n_normal = (df.label == 0).sum()
n_hiper  = (df.label == 1).sum()
ratio    = n_normal / n_hiper

print(f"Total de imagens  : {n_total:,}")
print(f"  Normal          : {n_normal:,}  ({100 * n_normal / n_total:.1f}%)")
print(f"  Hipertensiva    : {n_hiper:,}  ({100 * n_hiper / n_total:.1f}%)")
print(f"  Ratio neg:pos   : {ratio:.1f}:1")
print()
print("Distribuicao por fonte:")
src_table = (
    df.groupby(["source", "label"])
      .size()
      .rename("n")
      .rename({0: "normal", 1: "hipertensiva"}, level="label")
)
print(src_table.to_string())

### 3.2 Estrutura por Paciente e Controle de Data Leakage

Um problema clássico em datasets médicos é o **data leakage por paciente**: um mesmo paciente pode ter múltiplas imagens (olho esquerdo e direito, diferentes exames). Se o split treino/validação/teste for feito por imagem aleatoriamente, imagens do mesmo paciente podem aparecer simultaneamente no treino e no teste. Isso faz o modelo aprender a reconhecer características do paciente (como pigmentação do fundo ocular ou padrão vascular individual) em vez de aprender a reconhecer a patologia, resultando em métricas artificialmente infladas e um classificador que falha na prática clínica.

**Solução adotada:** o split é feito por `patient_id`, garantindo que todas as imagens de um paciente estejam exclusivamente em um único subconjunto (treino, validação ou teste). Os prefixos `brset_` e `odir_` no `patient_id` previnem colisões entre as duas fontes.

In [ ]:
imgs_por_paciente = df.groupby("patient_id").size()

print(f"Pacientes únicos       : {df.patient_id.nunique():,}")
print(f"Imagens / paciente     : média={imgs_por_paciente.mean():.2f}  máx={imgs_por_paciente.max()}")
print(f"Pacientes com > 1 img  : {(imgs_por_paciente > 1).sum():,}")
print()
print("Split DEVE ser por patient_id para evitar data leakage.")

### 3.2.1 Avaliação de Qualidade das Imagens (BRSET)

O BRSET fornece metadados de qualidade por imagem: `quality` (avaliação global), `focus` (nitidez), `illum` (iluminação) e `artifacts` (artefatos). Imagens classificadas como inadequadas introduzem ruído no treinamento — o modelo pode aprender a associar artefatos ou desfoque com a classe positiva, em vez de sinais clínicos reais.

**Critério adotado:** são mantidas apenas imagens com `quality == "Adequate"`. Imagens do ODIR-5K não possuem metadados de qualidade e são mantidas integralmente.

In [ ]:
brset_df = df[df.source == "brset"]
print("=== quality ===")
print(brset_df["quality"].value_counts())
print("\n=== focus ===")
print(brset_df["focus"].value_counts())
print("\n=== illum ===")
print(brset_df["illum"].value_counts())
print("\n=== qualidade por label ===")
print(brset_df.groupby(["label", "quality"]).size())

### 3.3 Distribuição de Classes e Fontes

O desbalanceamento de classes é o principal desafio técnico deste trabalho. Com uma razão de aproximadamente 18:1 entre imagens normais e hipertensivas, um classificador ingênuo que preveja sempre "normal" atingiria acurácia acima de 94% — uma métrica enganosa. Por isso, utilizamos AUC-ROC e AUC-PR como métricas primárias, que são invariantes à distribuição de classes.

**Estratégias adotadas para mitigar o desbalanceamento:**

1. **WeightedRandomSampler**: durante o treinamento, cada amostra é sorteada com probabilidade inversamente proporcional à frequência de sua classe, aproximando a distribuição dos batches de 1:1. Isso permite que o modelo veja exemplos positivos com a mesma frequência que negativos em cada epoch, sem duplicar fisicamente os dados.

2. **Focal Loss** (Lin et al., 2017): modifica a binary cross-entropy adicionando um fator `(1 - pₜ)^γ` que reduz o peso de exemplos fáceis (i.e., amostras da classe majoritária que o modelo já classifica com alta confiança). O parâmetro `α = 0.75` adiciona um peso extra à classe positiva rara; `γ = 2.0` é o valor canônico proposto pelos autores, que demonstrou bom comportamento em cenários de detecção com desbalanceamento severo.

3. **Aumento de dados**: aplicado exclusivamente ao conjunto de treino para aumentar a diversidade artificial de exemplos positivos.

In [ ]:
import matplotlib.pyplot as plt

label_names = {0: "Normal", 1: "Hipertensiva"}
palette     = {"Normal": "#4C9BE8", "Hipertensiva": "#E85C4C"}

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
fig.suptitle("Distribuição do Dataset", fontsize=14, fontweight="bold")

# Contagem total por classe
counts = df["label"].map(label_names).value_counts()
bars = axes[0].bar(
    counts.index, counts.values,
    color=[palette[k] for k in counts.index],
    edgecolor="white", linewidth=0.8, width=0.5
)
for bar, val in zip(bars, counts.values):
    axes[0].text(
        bar.get_x() + bar.get_width() / 2, bar.get_height() + max(counts.values) * 0.02,
        f"{val:,}", ha="center", va="bottom", fontsize=11, fontweight="bold"
    )
axes[0].set_title("Contagem por Classe", pad=10)
axes[0].set_ylabel("Num. de Imagens")
axes[0].set_ylim(0, max(counts.values) * 1.15)
axes[0].spines[["top", "right"]].set_visible(False)

# Contagem por fonte e classe
src_pivot = (
    df.assign(classe=df["label"].map(label_names))
      .groupby(["source", "classe"])
      .size()
      .unstack(fill_value=0)
)
src_pivot.plot(
    kind="bar", ax=axes[1],
    color=[palette[c] for c in src_pivot.columns],
    edgecolor="white", linewidth=0.8, rot=30
)
axes[1].set_title("Contagem por Fonte e Classe", pad=10)
axes[1].set_xlabel("Fonte", labelpad=8)
axes[1].set_ylabel("Num. de Imagens")
axes[1].spines[["top", "right"]].set_visible(False)
axes[1].legend(title="Classe", framealpha=0.8)

plt.tight_layout()
plt.savefig("eda_distribuicao.png", dpi=150, bbox_inches="tight")
plt.show()

### 3.4 Amostras Visuais

Inspeção qualitativa de imagens representativas de cada classe. Na retinopatia hipertensiva, os achados típicos incluem estreitamento arteriolar, cruzamentos arteriovenosos patológicos (sinal de Gunn), hemorragias em chama de vela, exsudatos duros e, em casos graves, edema de papila. A distinção visual pode ser sutil e dependente da gravidade da hipertensão, o que justifica o uso de backbones pré-treinados com grande capacidade de extração de características.

**Estratégia de aumento de dados adotada:**

| Transformação | Justificativa clínica |
|---|---|
| `HorizontalFlip` | Olho esquerdo e direito são espelhados; válido anatomicamente |
| `VerticalFlip` | Aumenta diversidade de orientação de vasos |
| `Rotate(±15°)` | Variação natural na angulação de captura |
| `RandomBrightnessContrast` | Variação de equipamentos e condições de iluminação |
| `HueSaturationValue` | Diferenças de calibração entre câmeras fundoscópicas |
| `GaussianBlur / GaussNoise` | Simula imperfeições ópticas e variação de sensor |

Todas as transformações são aplicadas apenas ao conjunto de treino. Validação e teste recebem apenas redimensionamento e normalização, para avaliação sob condições realistas.

In [ ]:
import cv2
import matplotlib.pyplot as plt

def show_samples(df, n=4, seed=42):
    label_names = {0: "Normal", 1: "Hipertensiva"}
    row_colors  = {0: "#4C9BE8", 1: "#E85C4C"}

    fig, axes = plt.subplots(2, n, figsize=(3 * n, 7))
    fig.suptitle(
        "Amostras: Normal vs Retinopatia Hipertensiva",
        fontsize=13, fontweight="bold"
    )

    for row, lab in enumerate([0, 1]):
        subset = df[df.label == lab].sample(
            min(n, (df.label == lab).sum()), random_state=seed
        )
        for col, (_, r) in enumerate(subset.iterrows()):
            img = cv2.cvtColor(cv2.imread(str(r.path)), cv2.COLOR_BGR2RGB)
            axes[row, col].imshow(img)
            axes[row, col].axis("off")
            axes[row, col].set_title(r.source, fontsize=8, color="#555555")
        axes[row, 0].set_ylabel(
            label_names[lab], fontsize=12,
            fontweight="bold", color=row_colors[lab],
            rotation=90, labelpad=12
        )

    plt.tight_layout()
    plt.savefig("eda_amostras.png", dpi=150, bbox_inches="tight")
    plt.show()

show_samples(df)

In [ ]:
# sanity check do Dataset

from retino.data.dataset import FundusDataset, make_sampler, get_loaders

ds = FundusDataset(df, split="train")
img, label = ds[0]

print(f"Shape da imagem: {img.shape}")          # [3, 224, 224]
print(f"Dtype:           {img.dtype}")          # torch.float32
print(f"Min/Max:         {img.min():.2f} / {img.max():.2f}")  # normalizado
print(f"Label:           {label.item()}")

### 3.5 Verificação do Pipeline de Dados

Antes do treinamento, validamos três componentes críticos do pipeline: (1) o `FundusDataset` retorna tensores com forma e tipo corretos; (2) o `WeightedRandomSampler` produz batches com distribuição próxima de 1:1; e (3) a `FocalLoss` computa valores numericamente estáveis e menores que a BCE padrão para exemplos fáceis.

In [ ]:
# Verificar o sampler 

from collections import Counter

sampler = make_sampler(df)

# simula 1 epoch (len(df) sorteios)
indices  = list(sampler)
labels   = df["label"].values
contagem = Counter(labels[i] for i in indices)

print(f"Sorteios por classe em 1 epoch:")
print(f"  Normal (0):       {contagem[0]:,}")
print(f"  Hipertensiva (1): {contagem[1]:,}")
print(f"  Ratio:            {contagem[0]/contagem[1]:.2f}:1  (esperado ~1:1)")

# Resultado esperado -> ratio proximo de 1:1 entre samples

In [ ]:
from retino.data.losses import FocalLoss
import torch

loss_fn = FocalLoss(alpha=0.75, gamma=2.0)

# Simula batch de 8 imagens: 7 normais, 1 hipertensiva
logits  = torch.randn(8)
targets = torch.tensor([0, 0, 0, 0, 0, 0, 0, 1], dtype=torch.float32)

loss = loss_fn(logits, targets)
bce  = torch.nn.functional.binary_cross_entropy_with_logits(logits, targets)

print(f"Focal Loss : {loss.item():.4f}")
print(f"BCE padrao : {bce.item():.4f}")
print(f"Focal < BCE: {loss.item() < bce.item()}")
print()
print("Nota: Focal Loss < BCE indica que o fator (1-pt)^gamma esta")
print("atenuando corretamente os exemplos faceis (maioria normal).")